## Neural Networks and LLMs

In [1]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [19]:
LITE_MODE = True

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [20]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/735 [00:00<?, ?B/s]

c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\nraja\.cache\huggingface\hub\datasets--ed-donner--items_lite. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


data/train-00000-of-00001.parquet:   0%|          | 0.00/6.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/304k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


## Now we will compare the error rate 

Human?

Neural Network?

Gemini?

## Human

human_in.csv

    ↓

Descriptions + 0

0 means no humans predictions yet



human_out.csv

    ↓

Same descriptions + human guessed prices

In [21]:
with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [22]:
human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [23]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [24]:
human = human_pricer(test[99])
actual = test[99].price
print(f"Human predicted {human} for an item that actually costs {actual}")

Human predicted 220.0 for an item that actually costs 574.99


In [25]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

## Neural Network from scratch using Pytorch

In [26]:
# Prepare our documents and prices
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [27]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [28]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [29]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [30]:
# Define loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 2 # We will train the data 2 times

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 17091.439, Val Loss: 19761.143


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 17878.773, Val Loss: 18035.289


In [34]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [36]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$87 $52 $17 $2 $43 $179 $26 $65 $32 $39 $457 $136 $70 $164 $27 $12 $6 $10 $101 $2 $48 $8 $48 $2 $286 $276 $241 $28 $27 $42 $39 $141 $35 $27 $116 $273 $33 $89 $131 $67 $162 $88 $9 $110 $122 $44 $70 $44 $26 $37 $9 $60 $86 $34 $111 $41 $35 $157 $22 $29 $71 $6 $45 $4 $484 $71 $24 $263 $13 $242 $6 $29 $121 $105 $12 $40 $28 $40 $22 $64 $74 $153 $26 $48 $11 $35 $56 $160 $109 $111 $16 $64 $18 $13 $29 $86 $35 $25 $148 $252 $21 $74 $1 $56 $52 $61 $85 $254 $7 $124 $18 $71 $102 $26 $56 $138 $135 $39 $63 $44 $22 $236 $10 $18 $95 $14 $19 $152 $41 $57 $33 $118 $134 $22 $74 $23 $110 $65 $42 $77 $36 $178 $22 $129 $202 $73 $45 $314 $90 $7 $19 $199 $3 $83 $29 $139 $150 $12 $16 $12 $100 $12 $4 $25 $518 $15 $79 $12 $7 $37 $13 $17 $285 $36 $14 $30 $16 $3 $45 $98 $359 $13 $121 $32 $22 $85 $33 $34 $29 $11 $31 $52 $53 $36 $20 $11 $60 $27 $12 $12 

## Frontier Models - LLMs

Chatgpt

Gemini

In [37]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [47]:
print(test[748].summary)

Title: Jake's 3" Spindle Lift Kit EZGO TXT 1994-2001.5  
Category: Golf Cart Parts  
Brand: Golf Cart King  
Description: A bolt‑on 3‑inch spindle lift kit for EZGO TXT golf carts from 1994 to 2001.5.  
Details: Features cast spindles for superior strength, includes all necessary bolts and detailed installation instructions, and lifts the rear by 3 inches without cutting or welding.


In [48]:
messages_for(test[748])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Jake\'s 3" Spindle Lift Kit EZGO TXT 1994-2001.5  \nCategory: Golf Cart Parts  \nBrand: Golf Cart King  \nDescription: A bolt‑on 3‑inch spindle lift kit for EZGO TXT golf carts from 1994 to 2001.5.  \nDetails: Features cast spindles for superior strength, includes all necessary bolts and detailed installation instructions, and lifts the rear by 3 inches without cutting or welding.'}]

In [101]:
os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

## OpenAI

In [128]:
# The function for gpt-4.1-nano
def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gpt_4__1_nano, test)

In [129]:
# The function for gpt-5.1
def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)

## Gemini

In [126]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
evaluate(gemini_2__5_flash_lite, test, size = 10)